In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import sys
print(sys.executable)

/usr/bin/python3


Task 1: Data Ingestion & Exploration
Objective


Objective
Load the dataset into PySpark and perform exploratory analysis.


Requirements

Create Spark Session


In [5]:
from pyspark.sql import SparkSession
spark=SparkSession.builder\
    .appName("DataAnalysis")\
    .getOrCreate()

Load DataSet

In [51]:
df=spark.read.csv("/content/drive/MyDrive/BNPParibas_Data.csv",
header=True,inferSchema=True)

Display:


Schema

In [7]:
df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- tenure_months: integer (nullable = true)
 |-- monthly_charges: double (nullable = true)
 |-- total_charges: double (nullable = true)
 |-- contract_type: string (nullable = true)
 |-- internet_service: string (nullable = true)
 |-- support_tickets: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- churn: integer (nullable = true)



Record Count


In [8]:
df.count()

1000

Null Count


In [9]:
from pyspark.sql.functions import col,when,count
df.select([count(
    when(col(c).isNull(),c)
    ).alias(c)
    for c in df.columns
    ]).show()


+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges|contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|          0|  0|            0|              0|            0|            0|               0|              0|             0|    0|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+



Duplicate Count


In [10]:
duplicate_count=df.count()-df.distinct().count()
duplicate_count

0

Data Types

In [11]:
df.dtypes

[('customer_id', 'int'),
 ('age', 'int'),
 ('tenure_months', 'int'),
 ('monthly_charges', 'double'),
 ('total_charges', 'double'),
 ('contract_type', 'string'),
 ('internet_service', 'string'),
 ('support_tickets', 'int'),
 ('payment_method', 'string'),
 ('churn', 'int')]

Task 2: ETL Pipeline Development

Objective
Build a complete ETL pipeline.


Extract
Read source dataset.


In [12]:
df=spark.read.csv(
    "/content/drive/MyDrive/BNPParibas_Data.csv",
    header=True,
    inferSchema=True
)

Transform

Perform:



Missing Value Treatment


In [13]:
from pyspark.sql.functions import col,when,count
df.select([count(
    when(col(c).isNull(),c)
    ).alias(c)
    for c in df.columns
    ]).show()

+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|customer_id|age|tenure_months|monthly_charges|total_charges|contract_type|internet_service|support_tickets|payment_method|churn|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+
|          0|  0|            0|              0|            0|            0|               0|              0|             0|    0|
+-----------+---+-------------+---------------+-------------+-------------+----------------+---------------+--------------+-----+



In [14]:
df=df.fillna({"internet_service":"Unknown"})

Duplicate Removal

In [15]:
df=df.drop_duplicates()

Data Type Conversion

In [16]:
df.dtypes  # already correct

[('customer_id', 'int'),
 ('age', 'int'),
 ('tenure_months', 'int'),
 ('monthly_charges', 'double'),
 ('total_charges', 'double'),
 ('contract_type', 'string'),
 ('internet_service', 'string'),
 ('support_tickets', 'int'),
 ('payment_method', 'string'),
 ('churn', 'int')]

Feature Engineering

Average monthly spend

In [17]:
df=df.withColumn("average_monthly_spend",col("total_charges")/col("tenure_months"))
df.show(5)

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|average_monthly_spend|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------------+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|   61.974666666666664|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|    85.44131578947368|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|   28.330750000000002|
|        410| 19|           51|          88.61|      4368.39|      One Year|      

Aggregation

Find avg monthly charges 

In [18]:
from pyspark.sql.functions import avg
df.groupBy("internet_service")\
.agg(avg("monthly_charges")).alias('avg_monthly_charges').show()

+----------------+--------------------+
|internet_service|avg(monthly_charges)|
+----------------+--------------------+
|            None|   82.02269662921348|
|             DSL|   80.41257069408742|
|           Fiber|   79.28475095785436|
+----------------+--------------------+



Load:
Store transformed data in:


In [19]:
df.write.mode("overwrite").parquet("silver/")
silver_df = spark.read.parquet("silver/")
silver_df.show(5)

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|average_monthly_spend|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------------+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|   61.974666666666664|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|    85.44131578947368|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|   28.330750000000002|
|        410| 19|           51|          88.61|      4368.39|      One Year|      

Task 3: ELT Pipeline & Medallion Architecture


In [20]:
# Objective
# Implement Bronze-Silver-Gold architecture.
# Bronze Layer
# Raw Data
# bronze/

broze_df=spark.read.csv("/content/drive/MyDrive/BNPParibas_Data.csv",
header=True,inferSchema=True)

broze_df.write.mode('overwrite').parquet('bronze/')


In [21]:
# Silver Layer
# Cleaned Data
# silver/

from pyspark.sql.functions import *
silver=broze_df.drop_duplicates()

silver=silver.fillna(0,subset=['monthly_charges','total_charges',
    'support_tickets'])

silver=silver.withColumn('Customer_Status',when(col('churn')==1,'Churned').otherwise('Active'))
silver.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|Customer_Status|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|        Churned|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|        Churned|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|         Active|
|        410| 19|           51|          88.61|      4368.39|      One Year|           Fiber|              0|         

In [22]:
silver.write.mode('overwrite').parquet('/content/drive/MyDrive/silver/')


In [23]:
# Gold Layer
# Business Ready Dataset
# gold/

# Category performance

kpi1 = silver.groupBy("contract_type").agg(count("*").alias("customers"),sum("churn").alias("churned_customers"))
kpi1.show()

kpi1.write.mode('overwrite').parquet('/content/drive/MyDrive/gold/curn_by_contract_type')

+--------------+---------+-----------------+
| contract_type|customers|churned_customers|
+--------------+---------+-----------------+
|Month-to-Month|      583|              405|
|      One Year|      278|               67|
|      Two Year|      139|               30|
+--------------+---------+-----------------+



In [24]:
# KPI 2: Revenue Analysis by Internet Service

kpi2=silver.groupBy('internet_service').agg(sum('total_charges')).alias('total_revenue')
kpi2.show()

kpi2.write.mode('overwrite').parquet('/content/drive/MyDrive//gold/revenue_using_internet_service')

+----------------+------------------+
|internet_service|sum(total_charges)|
+----------------+------------------+
|            None|         288972.56|
|             DSL|         1121281.7|
|           Fiber|1389981.1199999992|
+----------------+------------------+



In [25]:
#KPI 3: Customer Analysis by payment method
kpi3=silver.groupBy('payment_method').count()
kpi3.show()
kpi3.write.mode('overwrite').parquet('/content/drive/MyDrive//gold/customer_analysis_using_payment_method')

+--------------+-----+
|payment_method|count|
+--------------+-----+
|   Credit Card|  279|
| Bank Transfer|  218|
|          Cash|  243|
|           UPI|  260|
+--------------+-----+



Task 4: PySpark + Pandas Integration

In [26]:
# Objective
# Demonstrate interoperability between Pandas and Spark.

# Requirements
# Convert Spark → Pandas
# df.toPandas()

pandas_convert=silver.toPandas()
pandas_convert.head()



,customer_id,age,tenure_months,monthly_charges,total_charges,contract_type,internet_service,support_tickets,payment_method,churn,Customer_Status
0,1,56,15,59.23,929.62,Month-to-Month,Fiber,5,UPI,1,Churned
1,182,23,38,80.86,3246.77,Month-to-Month,DSL,3,Cash,1,Churned
2,199,20,40,29.91,1133.23,Two Year,Fiber,3,Credit Card,0,Active
3,410,19,51,88.61,4368.39,One Year,Fiber,0,UPI,0,Active
4,664,56,47,141.36,6544.26,Month-to-Month,DSL,1,Bank Transfer,1,Churned


Feature Engineering in Pandas

In [27]:
#Ratio Column
pandas_convert['ratio_of_charges']=pandas_convert['total_charges']/(pandas_convert['tenure_months']+1)
pandas_convert.head()

,customer_id,age,tenure_months,monthly_charges,total_charges,contract_type,internet_service,support_tickets,payment_method,churn,Customer_Status,ratio_of_charges
0,1,56,15,59.23,929.62,Month-to-Month,Fiber,5,UPI,1,Churned,58.101250
1,182,23,38,80.86,3246.77,Month-to-Month,DSL,3,Cash,1,Churned,83.250513
2,199,20,40,29.91,1133.23,Two Year,Fiber,3,Credit Card,0,Active,27.639756
3,410,19,51,88.61,4368.39,One Year,Fiber,0,UPI,0,Active,84.007500
4,664,56,47,141.36,6544.26,Month-to-Month,DSL,1,Bank Transfer,1,Churned,136.338750


In [28]:
#Percentage Column
pandas_convert['revenue_percentage']=(pandas_convert['total_charges']/(pandas_convert['total_charges'].sum()))*100
pandas_convert.head()

,customer_id,age,tenure_months,monthly_charges,total_charges,contract_type,internet_service,support_tickets,payment_method,churn,Customer_Status,ratio_of_charges,revenue_percentage
0,1,56,15,59.23,929.62,Month-to-Month,Fiber,5,UPI,1,Churned,58.101250,0.033198
1,182,23,38,80.86,3246.77,Month-to-Month,DSL,3,Cash,1,Churned,83.250513,0.115946
2,199,20,40,29.91,1133.23,Two Year,Fiber,3,Credit Card,0,Active,27.639756,0.040469
3,410,19,51,88.61,4368.39,One Year,Fiber,0,UPI,0,Active,84.007500,0.156001
4,664,56,47,141.36,6544.26,Month-to-Month,DSL,1,Bank Transfer,1,Churned,136.338750,0.233704


In [29]:
#Groth Metric
pandas_convert['growth_metric']=((pandas_convert['monthly_charges']-pandas_convert['monthly_charges'].mean())/pandas_convert['monthly_charges'].mean())*100
pandas_convert.head()

,customer_id,age,tenure_months,monthly_charges,total_charges,contract_type,internet_service,support_tickets,payment_method,churn,Customer_Status,ratio_of_charges,revenue_percentage,growth_metric
0,1,56,15,59.23,929.62,Month-to-Month,Fiber,5,UPI,1,Churned,58.101250,0.033198,-25.932086
1,182,23,38,80.86,3246.77,Month-to-Month,DSL,3,Cash,1,Churned,83.250513,0.115946,1.116521
2,199,20,40,29.91,1133.23,Two Year,Fiber,3,Credit Card,0,Active,27.639756,0.040469,-62.597141
3,410,19,51,88.61,4368.39,One Year,Fiber,0,UPI,0,Active,84.007500,0.156001,10.808001
4,664,56,47,141.36,6544.26,Month-to-Month,DSL,1,Bank Transfer,1,Churned,136.338750,0.233704,76.772587


In [30]:
# Convert Pandas → Spark
# spark.createDataFrame(pdf)

spark_convert=spark.createDataFrame(pandas_convert)
spark_convert.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+------------------+--------------------+-------------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|Customer_Status|  ratio_of_charges|  revenue_percentage|      growth_metric|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+------------------+--------------------+-------------------+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|        Churned|          58.10125| 0.03319792352598588|-25.932085862757386|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|        Churned| 83.25051282051282| 0.1159

Task 5: Spark SQL

Generate analytical reports using Spark SQL.

In [31]:
# Requirements
# Create Temporary Views.
# Execute minimum 5 SQL Queries.

silver.createOrReplaceTempView('customers')



In [32]:
# Query 1
# Top Categories 
#top contract type by customer count

spark.sql("""select contract_type,count(*) as total_customers from customers
          group by contract_type order by total_customers """).show()



+--------------+---------------+
| contract_type|total_customers|
+--------------+---------------+
|      Two Year|            139|
|      One Year|            278|
|Month-to-Month|            583|
+--------------+---------------+



In [33]:
# Query 2
# Highest Revenue Segment
spark.sql("""select internet_service,sum(total_charges) as revenue from customers
         group by internet_service order by revenue desc""").show()


+----------------+------------------+
|internet_service|           revenue|
+----------------+------------------+
|           Fiber|1389981.1199999992|
|             DSL|         1121281.7|
|            None|         288972.56|
+----------------+------------------+



In [34]:
# Query 3
# Average Metric by Group

spark.sql("""select contract_type,avg(monthly_charges) as average_charges from customers
          group by contract_type order by average_charges desc""").show()

+--------------+-----------------+
| contract_type|  average_charges|
+--------------+-----------------+
|Month-to-Month|80.42109777015438|
|      One Year|80.04780575539571|
|      Two Year|77.90187050359717|
+--------------+-----------------+



In [35]:
# Query 4
# Monthly Trend Analysis
# top 10 customer with highest total charges

spark.sql("""select customer_id,total_charges from customers 
          order by total_charges desc limit 10 """).show()


+-----------+-------------+
|customer_id|total_charges|
+-----------+-------------+
|         95|     10772.52|
|        391|     10558.95|
|        179|     10445.05|
|        519|     10071.46|
|        940|      9940.84|
|        570|      9754.19|
|        939|      9571.84|
|        950|      9465.68|
|        501|      9457.37|
|        481|      9261.92|
+-----------+-------------+



Query 5
Top 10 Records by Business Metric


In [36]:
spark.sql("""select Customer_Status,sum(total_charges) as total_charges_paid from customers
        group by Customer_Status order by total_charges_paid desc """).show()

+---------------+------------------+
|Customer_Status|total_charges_paid|
+---------------+------------------+
|         Active|1401626.5299999996|
|        Churned|1398608.8499999978|
+---------------+------------------+



Task 6: Advanced Transformations


Objective


Window Functions
Implement:


In [37]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank,row_number,dense_rank,lag,lead,desc

Row_number

In [38]:
silver.withColumn('row_num',row_number().over(Window.orderBy('total_charges'))).show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+-------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|Customer_Status|row_num|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+-------+
|        277| 35|            1|          19.98|        19.63|      Two Year|           Fiber|              2|          Cash|    0|         Active|      1|
|         85| 25|            2|          19.47|        42.43|Month-to-Month|             DSL|              1|           UPI|    0|         Active|      2|
|        209| 61|            1|          52.88|        50.14|      One Year|           Fiber|              2|   Credit Card|    0|         Active|      3|
|        863| 67|            1|          54.64|        55.44|Month-to-

Rank

In [39]:
rank=silver.withColumn('rank_of_customer',rank().over(Window.orderBy(col('total_charges').desc()))
)
rank.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+----------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|Customer_Status|rank_of_customer|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+----------------+
|         95| 43|           69|         149.18|     10772.52|Month-to-Month|             DSL|              0|           UPI|    1|        Churned|               1|
|        391| 69|           69|         149.49|     10558.95|      One Year|           Fiber|              3|           UPI|    0|         Active|               2|
|        179| 19|           69|         145.22|     10445.05|Month-to-Month|             DSL|              4|          Cash|    0|         Active|               3|
|        519| 47

dense rank

In [40]:
dense_rank=silver.withColumn('dense_rank_of_costumer',dense_rank().over(Window.orderBy('total_charges')))
dense_rank.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+----------------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|Customer_Status|dense_rank_of_costumer|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+----------------------+
|        277| 35|            1|          19.98|        19.63|      Two Year|           Fiber|              2|          Cash|    0|         Active|                     1|
|         85| 25|            2|          19.47|        42.43|Month-to-Month|             DSL|              1|           UPI|    0|         Active|                     2|
|        209| 61|            1|          52.88|        50.14|      One Year|           Fiber|              2|   Credit Card|    0|         Active|    

In [41]:
lag_operation=silver.withColumn('lag_col',lag(col('monthly_charges')).over(Window.orderBy('monthly_charges')))
lag_operation.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+-------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|Customer_Status|lag_col|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+-------+
|        806| 50|           24|          10.22|       253.07|Month-to-Month|            None|              4| Bank Transfer|    0|         Active|   NULL|
|        400| 48|           17|          10.45|       167.94|      One Year|           Fiber|              1|   Credit Card|    0|         Active|  10.22|
|        380| 23|           21|          10.73|       226.72|      One Year|           Fiber|              0| Bank Transfer|    1|        Churned|  10.45|
|        773| 53|           60|          10.77|       660.38|      Two

In [42]:
lead_operation=silver.withColumn('lead_col',lead(col('monthly_charges')).over(Window.orderBy('monthly_charges')))
lead_operation.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+--------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|Customer_Status|lead_col|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+--------+
|        806| 50|           24|          10.22|       253.07|Month-to-Month|            None|              4| Bank Transfer|    0|         Active|   10.45|
|        400| 48|           17|          10.45|       167.94|      One Year|           Fiber|              1|   Credit Card|    0|         Active|   10.73|
|        380| 23|           21|          10.73|       226.72|      One Year|           Fiber|              0| Bank Transfer|    1|        Churned|   10.77|
|        773| 53|           60|          10.77|       660.38|   

Join Operations


Implement:


In [64]:
# Create a lookup table manually and join with the main dataset.
lookup_data=[("Premium", "High Value Customer"),
            ("Standard", "Medium Value Customer"),
            ("Basic", "Low Value Customer")
            ]
df=spark.createDataFrame(lookup_data,["customer_spending_level","description"])
df.show()

+-----------------------+--------------------+
|customer_spending_level|         description|
+-----------------------+--------------------+
|                Premium| High Value Customer|
|               Standard|Medium Value Cust...|
|                  Basic|  Low Value Customer|
+-----------------------+--------------------+



In [62]:


silver=silver.withColumn('customer_spending_level',when(col('monthly_charges')>80,'Premium')
                        .when(col('monthly_charges')>=60,'Standard')
                        .otherwise('Basic'))

silver.show()


+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+-----------------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|Customer_Status|customer_spending_level|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+-----------------------+
|          1| 56|           15|          59.23|       929.62|Month-to-Month|           Fiber|              5|           UPI|    1|        Churned|                  Basic|
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|        Churned|                Premium|
|        199| 20|           40|          29.91|      1133.23|      Two Year|           Fiber|              3|   Credit Card|    0|         Active

Inner Join

In [44]:
inner_join=silver.join(df,silver['customer_spending_level']==df['customer_spending_level'],'inner')
inner_join.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+-----------------------+-----------------------+-------------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|Customer_Status|customer_spending_level|customer_spending_level|        description|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+-----------------------+-----------------------+-------------------+
|        588| 43|           45|         147.69|      6701.92|      One Year|             DSL|              4| Bank Transfer|    1|        Churned|                Premium|                Premium|High Value Customer|
|        167| 56|            5|          94.88|       517.19|Month-to-Month|           Fiber|              0|           UPI|    0|         A

Left join

In [45]:
left_join=silver.join(df,silver['customer_spending_level']==df['customer_spending_level'],'left')
left_join.show()

+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+-----------------------+-----------------------+--------------------+
|customer_id|age|tenure_months|monthly_charges|total_charges| contract_type|internet_service|support_tickets|payment_method|churn|Customer_Status|customer_spending_level|customer_spending_level|         description|
+-----------+---+-------------+---------------+-------------+--------------+----------------+---------------+--------------+-----+---------------+-----------------------+-----------------------+--------------------+
|        182| 23|           38|          80.86|      3246.77|Month-to-Month|             DSL|              3|          Cash|    1|        Churned|                Premium|                Premium| High Value Customer|
|        410| 19|           51|          88.61|      4368.39|      One Year|           Fiber|              0|           UPI|    0|      

Task 8: Performance Optimization

Objective


Optimize Spark Jobs.

In [46]:
# Implement
# Cache
# df.cache()


silver.cache()

# Trigger cache
silver.count()

1000

In [47]:
# Persist
from pyspark import StorageLevel

silver.persist(StorageLevel.MEMORY_AND_DISK)

silver.count()

1000

In [48]:
# Repartition
silver_repartitioned = silver.repartition(8)

print("Partitions:", silver_repartitioned.rdd.getNumPartitions())

Partitions: 8


In [ ]:
# Broadcast Join
from pyspark.sql.functions import broadcast

joined_df = silver.join(
    broadcast(lookup_data),
    "customer_spending_level",
    "inner"
)

joined_df.show()

Measure Query Time

In [67]:
# Measure Query Time

import time
start = time.time()
silver.filter(col("monthly_charges") > 50).count()
end = time.time()
print("Query Time:", end - start, "seconds")


Query Time: 1.665064811706543 seconds


In [70]:
# Measure Aggregation Time
start = time.time()

silver.groupBy("contract_type").agg(
    avg("monthly_charges")
).show()

end = time.time()

print("Aggregation Time:", end - start, "seconds")

+--------------+--------------------+
| contract_type|avg(monthly_charges)|
+--------------+--------------------+
|Month-to-Month|   80.42109777015438|
|      One Year|   80.04780575539566|
|      Two Year|   77.90187050359714|
+--------------+--------------------+

Aggregation Time: 3.450037717819214 seconds


In [ ]:
# Measure Join Time
# Before Broadcast
start = time.time()

silver.join(
    lookup_data,
    "customer_spending_level",
    "inner"
).count()

end = time.time()

print("Normal Join Time:", end - start, "seconds")

In [ ]:
# After Broadcast
start = time.time()

silver.join(
    broadcast(lookup_data),
    "customer_spending_level",
    "inner"
).count()

end = time.time()

print("Broadcast Join Time:", end - start, "seconds")